# 04 · Model selection

Goal: an honest baseline without duplicate listings, then compare many
models (LazyPredict), then tune the best one (Optuna).

## 1. Honest baseline: remove duplicates

EDA found 925 duplicate listings (~10%). If copies land in both train and
test, the test score is too optimistic. Here the same model and pipeline
as in `02_explore_train_full_data` is trained twice: with and without
duplicates.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error

df = pd.read_csv("../data/item_clean_full.csv")
df.shape

(9541, 13)

In [3]:
def prepare_features(df):
    """Same feature preparation as in 02: group rare districts,
    one-hot encode, drop non-features, fill missing hasRepair."""
    df = df.copy()
    counts = df["location"].value_counts()
    common = counts[counts >= 5].index
    df["location"] = df["location"].where(df["location"].isin(common), "Other")
    df = pd.get_dummies(df, columns=["location"])

    features = df.drop(columns=["id", "price", "currency", "price_per_m2", "city"])
    features["hasRepair"] = features["hasRepair"].fillna(features["hasRepair"].mode()[0])
    target = df["price"]
    return features, target


def evaluate(model, features_test, target_test):
    predictions = model.predict(features_test)
    return {
        "MAE": mean_absolute_error(target_test, predictions),
        "RMSE": root_mean_squared_error(target_test, predictions),
        "R2": r2_score(target_test, predictions),
        "MAPE %": mean_absolute_percentage_error(target_test, predictions) * 100,
    }


def train_and_evaluate(df):
    features, target = prepare_features(df)
    features_train, features_test, target_train, target_test = train_test_split(
        features, target, test_size=0.2, random_state=42
    )
    model = RandomForestRegressor(n_estimators=50, random_state=42)
    model.fit(features_train, target_train)
    return evaluate(model, features_test, target_test)

In [4]:
feature_cols = ["price", "rooms", "area", "floor", "floors", "location", "hasRepair"]
df_dedup = df.drop_duplicates(subset=feature_cols)
print(len(df), "->", len(df_dedup))

results = pd.DataFrame({
    "with duplicates": train_and_evaluate(df),
    "without duplicates": train_and_evaluate(df_dedup),
}).T
results

9541 -> 8616


,MAE,RMSE,R2,MAPE %
with duplicates,42046.907644,96515.107795,0.784948,13.469270
without duplicates,44292.718016,112818.813618,0.736739,14.120557


**Findings:** Removing 925 duplicates (9541 → 8616 rows) makes every
metric worse: MAE 42,047 → 44,293 (+5%), RMSE 96,515 → 112,819 (+17%),
R² 0.785 → 0.737, MAPE 13.5% → 14.1%. The previous score was optimistic:
copies of the same listing were in both train and test. RMSE rose the
most, because many duplicates were large expensive listings that the
model had effectively memorized. From here on, all models are trained
and compared on the deduplicated data; the honest baseline is
MAE ≈ 44.3k AZN.

### Learning curve: would more data help?

Same model, same test set, trained on growing parts of the training data.
If MAE keeps falling at 100%, more data is likely to help.

In [5]:
features, target = prepare_features(df_dedup)
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)

for share in [0.25, 0.5, 0.75, 1.0]:
    n = int(len(features_train) * share)
    model = RandomForestRegressor(n_estimators=50, random_state=42)
    model.fit(features_train.iloc[:n], target_train.iloc[:n])
    mae = mean_absolute_error(target_test, model.predict(features_test))
    print(f"{n} rows: MAE {mae:,.0f}")

1723 rows: MAE 53,448
3446 rows: MAE 50,580
5169 rows: MAE 46,247
6892 rows: MAE 44,293


## 2. More data: full catalog

Full SearchItems scrape (58,675 listings) cleaned by `scripts/clean_data.py`
→ 44,382 rows (duplicates already removed there). 98.4% of listings are in
Baku; Xırdalan 612, Sumqayıt 83, other cities single listings.

In [6]:
df_all = pd.read_csv("../data/item_clean_all.csv")
print(df_all.shape)

results_all = pd.DataFrame({
    "8.6k rows (old, deduplicated)": train_and_evaluate(df_dedup),
    "44k rows (full catalog)": train_and_evaluate(df_all),
}).T
results_all

(44382, 13)


,MAE,RMSE,R2,MAPE %
"8.6k rows (old, deduplicated)",44292.718016,112818.813618,0.736739,14.120557
44k rows (full catalog),38150.570670,105929.263682,0.812386,11.749728


**Findings:** Same model, same pipeline, only more data: MAE
44.3k → 38.2k (−14%), R² 0.737 → 0.812, MAPE 14.1% → 11.7%. RMSE improved
less (−6%): ordinary apartments benefit most, large errors on rare
expensive listings largely remain. All following steps use the full
catalog (`item_clean_all.csv`).

In [7]:
features, target = prepare_features(df_all)
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)

for share in [0.25, 0.5, 0.75, 1.0]:
    n = int(len(features_train) * share)
    model = RandomForestRegressor(n_estimators=50, random_state=42)
    model.fit(features_train.iloc[:n], target_train.iloc[:n])
    mae = mean_absolute_error(target_test, model.predict(features_test))
    print(f"{n} rows: MAE {mae:,.0f}")

8876 rows: MAE 47,976
17752 rows: MAE 42,942
26628 rows: MAE 40,211
35505 rows: MAE 38,151


**Findings:** MAE still falls with more data, but with diminishing
returns (−5.0k, −2.7k, −2.1k per step). The full bina.az catalog is now
used, so further gains must come from the model and the features.

## 3. Compare many models (LazyPredict)

LazyPredict trains ~40 regressors with default settings. Some of them
(Gaussian process, kernel ridge, SVR) scale badly with rows, so the
comparison runs on a random sample: 8,000 train / 2,000 test rows taken
from the full-catalog split. The goal is a ranking, not final numbers:
the top models are then evaluated on the full data.

In [8]:
from lazypredict.Supervised import LazyRegressor

# Random subsets of the full-catalog split from section 2
features_train_s = features_train.sample(8000, random_state=42).astype(float)
target_train_s = target_train.loc[features_train_s.index]
features_test_s = features_test.sample(2000, random_state=42).astype(float)
target_test_s = target_test.loc[features_test_s.index]

reg = LazyRegressor(verbose=0, ignore_warnings=True, timeout=120,
                    custom_metric=mean_absolute_error)
models, predictions = reg.fit(features_train_s, features_test_s,
                              target_train_s, target_test_s)
models

,Adjusted R-Squared,R-Squared,RMSE,mean_absolute_error,Time Taken
Model,,,,,
HistGradientBoostingRegressor,7.604703e-01,7.712545e-01,9.876574e+04,4.902103e+04,0.537380
RandomForestRegressor,7.484467e-01,7.597723e-01,1.012142e+05,4.465650e+04,3.469968
GradientBoostingRegressor,7.428404e-01,7.544183e-01,1.023359e+05,5.315163e+04,0.903083
BaggingRegressor,7.351661e-01,7.470896e-01,1.038517e+05,4.620584e+04,0.406913
ExtraTreesRegressor,7.298094e-01,7.419741e-01,1.048967e+05,4.428704e+04,4.649332
LassoLarsIC,7.159982e-01,7.287847e-01,1.075442e+05,5.557944e+04,0.089504
LarsCV,7.159523e-01,7.287408e-01,1.075530e+05,5.560147e+04,0.213283
LassoLarsCV,7.159523e-01,7.287408e-01,1.075530e+05,5.560147e+04,0.176878
LassoCV,7.159361e-01,7.287254e-01,1.075560e+05,5.560590e+04,0.167384


In [9]:
pd.options.display.float_format = "{:,.3f}".format
models.sort_values("mean_absolute_error").head(10)

,Adjusted R-Squared,R-Squared,RMSE,mean_absolute_error,Time Taken
Model,,,,,
ExtraTreesRegressor,0.730,0.742,"104,896.692","44,287.039",4.649
RandomForestRegressor,0.748,0.760,"101,214.250","44,656.500",3.470
BaggingRegressor,0.735,0.747,"103,851.669","46,205.840",0.407
HistGradientBoostingRegressor,0.760,0.771,"98,765.743","49,021.027",0.537
HuberRegressor,0.704,0.718,"109,748.378","51,498.130",0.200
PassiveAggressiveRegressor,0.700,0.714,"110,471.392","51,524.456",0.669
GradientBoostingRegressor,0.743,0.754,"102,335.916","53,151.632",0.903
RANSACRegressor,0.693,0.707,"111,779.735","53,712.710",0.369
KNeighborsRegressor,0.683,0.698,"113,532.923","54,030.792",0.146


**Findings:** The ranking depends on the metric. By R²/RMSE the leader is
HistGradientBoosting (R² 0.771, RMSE 98.8k), but by MAE (our primary
metric) the leaders are ExtraTrees (44.3k) and RandomForest (44.7k);
HistGradientBoosting is 4th (49.0k). Boosting handles large errors
better, averaging ensembles are more accurate on a typical apartment.
Linear models plateau at R² ~0.73 / MAE ~55.6k: the relationship is
non-linear. SVR, MLP and kernel models fail with default settings.
Shortlist for the full data: RandomForest, ExtraTrees,
HistGradientBoosting.

## 4. Shortlist on the full data

5-fold cross-validation on all 44k rows, plus model size (GitHub limit
100 MB, Render free tier 512 MB RAM).

In [10]:
import pickle
import time
from sklearn.model_selection import KFold, cross_validate
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor

X = features.astype(float)
y = target
cv = KFold(n_splits=5, shuffle=True, random_state=42)

shortlist = {
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42),
}
scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
    "MAPE": "neg_mean_absolute_percentage_error",
}

rows = {}
for name, model in shortlist.items():
    start = time.time()
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring)
    model.fit(X, y)
    rows[name] = {
        "MAE": -scores["test_MAE"].mean(),
        "MAE std": scores["test_MAE"].std(),
        "RMSE": -scores["test_RMSE"].mean(),
        "R2": scores["test_R2"].mean(),
        "MAPE %": -scores["test_MAPE"].mean() * 100,
        "size MB": len(pickle.dumps(model)) / 1_000_000,
        "seconds": time.time() - start,
    }

pd.DataFrame(rows).T

,MAE,MAE std,RMSE,R2,MAPE %,size MB,seconds
RandomForest,"37,598.469",360.991,"96,403.530",0.827,11.798,318.121,40.655
ExtraTrees,"36,046.397",535.858,"94,514.972",0.834,11.439,481.725,72.363
HistGradientBoosting,"47,114.612",421.899,"108,128.666",0.783,14.915,0.365,6.564


### Quick check before Optuna

One sensible manual setting per path: a smaller ExtraTrees and a
stronger HistGradientBoosting. The winner is the one to tune.

In [11]:
def compare(models_to_test):
    """5-fold CV metrics + model size for each model in the dict."""
    rows = {}
    for name, model in models_to_test.items():
        start = time.time()
        scores = cross_validate(model, X, y, cv=cv, scoring=scoring)
        model.fit(X, y)
        rows[name] = {
            "MAE": -scores["test_MAE"].mean(),
            "MAE std": scores["test_MAE"].std(),
            "RMSE": -scores["test_RMSE"].mean(),
            "R2": scores["test_R2"].mean(),
            "MAPE %": -scores["test_MAPE"].mean() * 100,
            "size MB": len(pickle.dumps(model)) / 1_000_000,
            "seconds": time.time() - start,
        }
    return pd.DataFrame(rows).T


quick = {
    "ExtraTrees small": ExtraTreesRegressor(
        n_estimators=50, min_samples_leaf=3, random_state=42, n_jobs=-1
    ),
    "HistGB stronger": HistGradientBoostingRegressor(
        max_iter=1000, learning_rate=0.05, max_leaf_nodes=63,
        early_stopping=False, random_state=42
    ),
}
compare(quick)

,MAE,MAE std,RMSE,R2,MAPE %,size MB,seconds
ExtraTrees small,"39,186.473",646.561,"102,107.523",0.806,12.186,69.012,29.231
HistGB stronger,"41,350.041",391.997,"100,298.560",0.813,12.886,7.124,66.903


**Findings:** Shrinking ExtraTrees (50 trees, min_samples_leaf=3) cuts the
size 7× (482 → 69 MB) but costs ~3.1k MAE (36.0k → 39.2k), and it is still
above 50 MB. One sensible manual setting improved HistGradientBoosting by
~12% (MAE 47.1k → 41.4k); it already beats the small ExtraTrees on RMSE
and R² at 7 MB. Decision: tune HistGradientBoosting with Optuna; the
small ExtraTrees (MAE 39.2k, 69 MB) is the benchmark to beat and the
fallback.

## 5. Tune HistGradientBoosting (Optuna)

20% of the data is set aside as a hold-out and not touched during tuning,
so the final comparison is not biased by the tuning itself. Optuna tunes
on the remaining 80% with 3-fold CV and minimizes MAE.

In [12]:
X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_dev.shape, X_holdout.shape)

(35505, 90) (8877, 90)


In [13]:
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
cv3 = KFold(n_splits=3, shuffle=True, random_state=42)


def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
        "max_iter": trial.suggest_int("max_iter", 200, 1500, step=100),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 255, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 100, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-3, 10, log=True),
        "max_features": trial.suggest_float("max_features", 0.5, 1.0),
    }
    model = HistGradientBoostingRegressor(early_stopping=False, random_state=42, **params)
    scores = cross_val_score(model, X_dev, y_dev, cv=cv3,
                             scoring="neg_mean_absolute_error", n_jobs=-1)
    return -scores.mean()


study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40, timeout=1800, show_progress_bar=True)

print("best MAE (3-fold CV on dev):", round(study.best_value))
study.best_params

  0%|          | 0/40 [00:00<?, ?it/s]

best MAE (3-fold CV on dev): 39586


{'learning_rate': 0.03776689505815151,
 'max_iter': 1100,
 'max_leaf_nodes': 222,
 'min_samples_leaf': 5,
 'l2_regularization': 0.0018323505683299347,
 'max_features': 0.5371847150398951}

**Findings:** 40 Optuna trials (28 min). Best 3-fold CV MAE on the dev
set: 39,586. Best settings: many small steps (learning_rate 0.038,
1,100 iterations) with fairly complex trees (222 leaves). Several values
sit at the edge of the search ranges (min_samples_leaf = 5,
l2_regularization ≈ 0.002, max_leaf_nodes 222 of 255) and the best trial
was #37 of 40, so a wider or longer search might still help.

## 6. Final comparison on the hold-out

Both candidates are trained on the same dev set (80%) and evaluated once
on the hold-out (20%) that was not used for tuning or model choice.

In [14]:
final_candidates = {
    "ExtraTrees small": ExtraTreesRegressor(
        n_estimators=50, min_samples_leaf=3, random_state=42, n_jobs=-1
    ),
    "HistGB tuned": HistGradientBoostingRegressor(
        early_stopping=False, random_state=42, **study.best_params
    ),
}

rows = {}
for name, model in final_candidates.items():
    model.fit(X_dev, y_dev)
    metrics = evaluate(model, X_holdout, y_holdout)
    metrics["size MB"] = len(pickle.dumps(model)) / 1_000_000
    rows[name] = metrics

pd.DataFrame(rows).T

,MAE,RMSE,R2,MAPE %,size MB
ExtraTrees small,"40,215.533","114,875.687",0.779,12.237,56.069
HistGB tuned,"37,456.452","100,103.860",0.832,11.482,27.428


**Findings:** On the untouched hold-out the tuned HistGradientBoosting
beats the small ExtraTrees on every metric (MAE 37.5k vs 40.2k,
RMSE 100.1k vs 114.9k, R² 0.832 vs 0.779, MAPE 11.5% vs 12.2%) and is
half the size (27 MB vs 56 MB).

## Summary

- Duplicates inflated the old score; honest baseline: MAE ≈ 44.3k.
- Full-catalog data (44k rows): MAE 38.2k with the same model.
- LazyPredict: tree ensembles and boosting lead; linear models plateau.
- ExtraTrees is the most accurate untuned model but 482 MB: too large
  for GitHub (100 MB) and Render's free tier (512 MB RAM).
- Tuned HistGradientBoosting (Optuna) wins on the hold-out at 27 MB
  → chosen as the final model.

## 7. Final model

Retrained on all 44k rows with the tuned settings and saved for the API.

In [15]:
import joblib

final_model = HistGradientBoostingRegressor(
    early_stopping=False, random_state=42, **study.best_params
)
final_model.fit(X, y)
joblib.dump(final_model, "../models/model.pkl")

print(len(final_model.feature_names_in_), "features")

90 features
